# ManoVarta Aya DAIC Continue

This notebook mounts Drive, syncs the latest `main`, downloads the saved Aya adapter bundle, and continues extractor fine-tuning with `DAIC-WOZ` auxiliary supervision.

Checkpoints and reports are written directly into `MyDrive/ManoVartaOutputs` so the run can resume safely after interruptions.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile

drive.mount('/content/drive', force_remount=True)

REPO_DIR = Path('/content/ManoVarta')
REPO_URL = 'https://github.com/RitwijParmar/ManoVarta.git'
DRIVE_ROOT = Path('/content/drive/MyDrive')
OUT_ROOT = DRIVE_ROOT / 'ManoVartaOutputs'
ART_ROOT = Path('/content/aya_artifacts')
OUTER_ROOT = Path('/content/aya_outer')

def run(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, check=True)

def find_daic_root(root: Path) -> Path:
    direct = root / 'DAIC-WOZ'
    if direct.exists():
        return direct
    for name in [
        'DAIC-WOZ',
        'DAIC_WOZ',
        'DAIC',
        'daic',
        'DAIC-WOZ_Depression',
        'DAICWOZ',
        'DAIC-WOZ_Depression_Dataset',
    ]:
        matches = sorted(root.rglob(name))
        if matches:
            return matches[0]
    csvs = sorted(root.rglob('train_split_Depression_AVEC2017.csv'))
    if csvs:
        return csvs[0].parent
    raise FileNotFoundError('Could not find DAIC-WOZ in MyDrive')

if (REPO_DIR / '.git').exists():
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
    run(['git', '-C', str(REPO_DIR), 'checkout', 'main'])
    run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'])
elif REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])

run([sys.executable, 'tools/colab_bootstrap.py'], cwd=REPO_DIR)
run(['nvidia-smi'])

from huggingface_hub import hf_hub_download

ART_ROOT.mkdir(parents=True, exist_ok=True)
if OUTER_ROOT.exists():
    shutil.rmtree(OUTER_ROOT)
OUTER_ROOT.mkdir(parents=True, exist_ok=True)

try:
    bundle_path = Path(
        hf_hub_download(
            repo_id='ritwijar/manovarta-aya-eval-artifacts',
            filename='aya_eval_upload.tar.gz',
            repo_type='model',
            local_dir=str(ART_ROOT),
        )
    )
except Exception as exc:
    raise RuntimeError(
        'Aya adapter download failed. If Hugging Face prompts for access, log in and rerun this cell.'
    ) from exc

with tarfile.open(bundle_path, 'r:gz') as tf:
    tf.extractall(OUTER_ROOT)

adapter_dir = OUTER_ROOT / 'aya_bundle'
assert (adapter_dir / 'adapter_config.json').exists(), f'Missing Aya adapter under {adapter_dir}'

daic_root = find_daic_root(DRIVE_ROOT)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    'tools/run_colab_daic_continue.py',
    '--device', 'cuda',
    '--daic-root', str(daic_root),
    '--init-adapter', str(adapter_dir),
    '--drive-dir', str(OUT_ROOT),
    '--extractor-model', 'CohereLabs/aya-expanse-8b',
]

print('DAIC root:', daic_root)
print('Aya adapter:', adapter_dir)
print('Running:', ' '.join(map(str, cmd)))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
